# 01 - Data Cleaning, OCR Risalah & Pembagian Dataset 80:20

Pipeline:
1. OCR seluruh PDF risalah → simpan teks ke `dataset/02_extracted/ocr_risalah/`
2. Pasangkan transkripsi Whisper (source) dengan teks OCR (target)
3. Bersihkan & validasi pasangan data
4. Split otomatis 80% latih / 20% uji → `train.csv` & `test.csv`

## 1. Import Library

In [1]:
import pandas as pd
import re
from pathlib import Path
import sys
sys.path.append('..')
from modules.ocr_risalah import proses_semua_pdf

## 2. OCR Semua Risalah PDF

In [2]:
DIR_PDF    = Path('../dataset/01_raw/risalah_pdf')
DIR_OCR    = Path('../dataset/02_extracted/ocr_risalah')

# Sesuaikan pola_awal dan pola_akhir dengan format risalah kamu!
POLA_AWAL  = r'MENYANYIKAN LAGU INDONESIA RAYA'
POLA_AKHIR = r'RAPAT DITUTUP PUKUL'

hasil_ocr = proses_semua_pdf(
    direktori_pdf=DIR_PDF,
    direktori_output=DIR_OCR,
    pola_awal=POLA_AWAL,
    pola_akhir=POLA_AKHIR,
)
print(f'Total risalah berhasil di-OCR: {len(hasil_ocr)}')

✅ Paripurna_Ke_10_Persidangan_II_2024_2025.pdf → 4094 kata diekstrak


KeyboardInterrupt: 

## 3. Load Transkripsi Whisper & Pasangkan dengan OCR

In [5]:
DIR_TRANSKRIP = Path('../dataset/02_extracted/whisper_transcripts')

# Ambil 400 kata pertama saja biar sinkronisasinya terjamin 100%
MAX_WORDS = 400 

baris = []
for txt_file in sorted(DIR_TRANSKRIP.glob('*.txt')):
    nama_tanpa_ext = txt_file.stem
    ocr_file = DIR_OCR / (nama_tanpa_ext + '.txt')
    
    if not ocr_file.exists():
        print(f'[SKIP] Tidak ada pasangan OCR untuk: {txt_file.name}')
        continue
        
    source = txt_file.read_text(encoding='utf-8').strip()
    target = ocr_file.read_text(encoding='utf-8').strip()
    
    if not source or not target:
        continue

    source_words = source.split()
    target_words = target.split()

    # --- JURUS AMBIL AWALNYA SAJA ---
    # Pastikan file punya setidaknya sekian kata, kalau terlalu pendek di-skip
    if len(source_words) >= 100 and len(target_words) >= 100:
        s_chunk = " ".join(source_words[:MAX_WORDS])
        t_chunk = " ".join(target_words[:MAX_WORDS])
        
        baris.append({'source': s_chunk, 'target': t_chunk})

df = pd.DataFrame(baris)
print(f'\n💎 Total data latih KUALITAS TINGGI (Hanya Awal Rapat): {len(df)} baris!')
df.head()


💎 Total data latih KUALITAS TINGGI (Hanya Awal Rapat): 30 baris!


,source,target
0,Hadirin sekalian. Rami persilakan untuk duduk ...,"Hadirin sekalian, kami persilakan untuk duduk ..."
1,Hadirin kami persilakan untuk duduk kembali. S...,"Hadirin, kami persilakan untuk duduk kembali. ..."
2,Selanjutnya kepada hadirin sekalian untuk dapa...,Selanjutnya kepada Hadirin sekalian untuk dapa...
3,Hadirin kami bersilakan untuk duduk kembali. S...,Kami persilahkan untuk duduk kembali. Sesuai d...
4,Hadirin kami persilakan untuk duduk kembali. S...,"Hadirin, kami persilakan untuk duduk Kembali. ..."


## 4. Validasi & Bersihkan

In [6]:
# Hapus baris kosong
df = df.dropna(subset=['source', 'target']).reset_index(drop=True)

# Filter pasangan yang terlalu pendek (Mencegah potongan sisa di akhir paragraf yang cuma 2-3 kata)
# Kita set minimal 50 kata agar model tetap punya konteks kalimat yang jelas.
df = df[df['source'].str.split().str.len() >= 50].reset_index(drop=True)
df = df[df['target'].str.split().str.len() >= 50].reset_index(drop=True)

print(f'Data valid siap latih setelah cleaning: {len(df)} baris')

# Mari kita lihat buktinya! Maksimal kata sekarang pasti tidak akan lebih dari 400.
print("\nStatistik Panjang Kata (Source & Target):")
print(df[['source', 'target']].apply(lambda c: c.str.split().str.len()).describe())

Data valid siap latih setelah cleaning: 30 baris

Statistik Panjang Kata (Source & Target):
       source  target
count    30.0    30.0
mean    400.0   400.0
std       0.0     0.0
min     400.0   400.0
25%     400.0   400.0
50%     400.0   400.0
75%     400.0   400.0
max     400.0   400.0


## 5. Split 80:20 → train.csv & test.csv

> Tidak ada validation set. Data dibagi langsung menjadi **data latih (80%)** dan **data uji (20%)**.

In [7]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f'Data latih (train): {len(df_train)} ({len(df_train)/len(df)*100:.0f}%)')
print(f'Data uji  (test) : {len(df_test)}  ({len(df_test)/len(df)*100:.0f}%)')

Data latih (train): 24 (80%)
Data uji  (test) : 6  (20%)


## 6. Simpan ke CSV

In [8]:
DATA_DIR = Path('../dataset/03_paired')
DATA_DIR.mkdir(parents=True, exist_ok=True)

df_train.to_csv(DATA_DIR / 'train.csv', index=False, encoding='utf-8')
df_test.to_csv(DATA_DIR  / 'test.csv',  index=False, encoding='utf-8')

print('Dataset berhasil disimpan:')
print(f'  train.csv → {len(df_train)} baris')
print(f'  test.csv  → {len(df_test)} baris')

Dataset berhasil disimpan:
  train.csv → 24 baris
  test.csv  → 6 baris
